In [13]:
import pandas as pd
from transformers import AutoTokenizer

In [14]:
tokenizer = AutoTokenizer.from_pretrained("nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16")

In [15]:
data = pd.read_csv("../data/generated/train_with_syntetic.csv")

In [16]:
data["token_len"] = data.generated_cot.apply(tokenizer.encode).apply(len)
data["token_len"].describe()

count    24393.000000
mean      1969.195589
std       1905.322361
min         28.000000
25%        650.000000
50%       1387.000000
75%       2278.000000
max       6854.000000
Name: token_len, dtype: float64

In [17]:
data.groupby("label").token_len.describe()

,count,mean,std,min,25%,50%,75%,max
label,,,,,,,,
bit manipulation,6947.0,3984.652080,2397.059850,28.0,34.0,5174.0,5586.5,6854.0
conversion to diff numeral system,1418.0,156.760931,25.305640,111.0,133.0,157.0,175.0,225.0
cryptarithm,6728.0,1978.361177,268.164851,1230.0,1783.0,1978.5,2165.0,2787.0
encryption,5587.0,801.782710,97.237395,486.0,737.0,801.0,868.0,1168.0
equations transformation,841.0,1336.623068,614.005108,194.0,834.0,1229.0,1890.0,2603.0
gravitational,1437.0,601.066110,48.139644,524.0,545.0,600.0,655.0,670.0
unit conversion,1435.0,246.074564,34.009341,188.0,224.0,237.0,276.0,307.0


In [6]:
data

,id,prompt,answer,prompt_eda,label,generated_cot,computed_answer,is_correct,token_len
0,00066667,"In Alice's Wonderland, a secret bit manipulati...",10010111,"In Alice's Wonderland, a secret bit manipulati...",bit manipulation,AND\n01 10 00010000 1\n12 21 00010000 1\n23 32...,10010111,True,4837
1,000b53cf,"In Alice's Wonderland, a secret bit manipulati...",01000011,"In Alice's Wonderland, a secret bit manipulati...",bit manipulation,AND\n01 10 000000000 a\n12 21 001000100 2 matc...,00011011,False,4952
2,0031df9c,"In Alice's Wonderland, a secret bit manipulati...",00110100,"In Alice's Wonderland, a secret bit manipulati...",bit manipulation,Macro pattern detected.\nOperation perfectly s...,00110100,True,30
3,004ef7c7,"In Alice's Wonderland, a secret bit manipulati...",11111111,"In Alice's Wonderland, a secret bit manipulati...",bit manipulation,AND\n01 10 11000000 2\n12 21 10000000 1\n23 32...,11110111,False,4806
4,00754598,"In Alice's Wonderland, a secret bit manipulati...",11101111,"In Alice's Wonderland, a secret bit manipulati...",bit manipulation,AND\n01 10 000100000 1\n12 21 000010000 1\n23 ...,11101111,True,4981
...,...,...,...,...,...,...,...,...,...
9495,fef61ca1,"In Alice's Wonderland, a secret set of transfo...",111,"In Alice's Wonderland, a secret set of transfo...",equations transformation,"First, we examine the operators in the equatio...",111,True,885
9496,ff0e37ae,"In Alice's Wonderland, a secret set of transfo...","%'""`","In Alice's Wonderland, a secret set of transfo...",equations transformation,"First, we examine the operators in the equatio...","%'""`",True,1805
9497,ff121f08,"In Alice's Wonderland, a secret set of transfo...",#<,"In Alice's Wonderland, a secret set of transfo...",equations transformation,"First, we examine the operators in the equatio...",#<,True,1841
9498,ff8409cd,"In Alice's Wonderland, a secret set of transfo...",9,"In Alice's Wonderland, a secret set of transfo...",equations transformation,"First, we examine the operators in the equatio...",9,True,404


In [7]:
target_labels = ['equations transformation']


correct_col = 'is_correct' if 'is_correct' in data.columns else 'is correct'

filtered_data = data[(data['label'].isin(target_labels)) & (data[correct_col] == True)]

sampled_data = filtered_data.groupby('label').sample(n=5, random_state=24424).reset_index(drop=True)

for index, row in sampled_data.iterrows():
    print(f"=== Категория: {row['label']} | ID: {row['id']} ===")
    print("--- Промпт (начало) ---")
    print(str(row['prompt']) + "...\n")
    
    print("--- Решение ---")
    cot_val = row.get('generated_cot', row.get('generated cot', 'Отсутствует'))
    print(cot_val)
    #
    #print("\n--- Вычисленный ответ ---")
    #ans_val = row.get('computed_answer', row.get('computed answer', 'Отсутствует'))
    #print(ans_val)
#
    #print("\n--- Истинный ответ ---")
    #ans_val = row.get('answer', row.get('answer', 'Отсутствует'))
    #print(ans_val)
    #
    #print("\n" + "="*80 + "\n")

=== Категория: equations transformation | ID: 9aa8dc92 ===
--- Промпт (начало) ---
In Alice's Wonderland, a secret set of transformation rules is applied to equations. Below are a few examples:
\|+<# = \|<#
#:+#: = #:#:
/@-{/ = -\"
Now, determine the result for: /#+</...

--- Решение ---
First, we examine the operators in the equations.
Let's look at the target expression and its operator, which is usually located in the middle.
There are no digits in the examples. This implies it is a cryptarithm; we need to decode the encrypted operations and values to calculate the result.
--- Решение ---
We model this as a cryptarithm: every visible non-operator symbol is one unique decimal digit, and multi-digit values cannot start with zero.
Parsed examples: '\\|+<#=\\|<#', '#:+#:=#:#:', '/@-{/=-\\"'
Target expression: '/#+</'
Symbols to decode: '"', '#', '/', ':', '<', '@', '\\', '{', '|'

Rule search
Evaluating operator '+'
Examples: \|+<#=\|<#, #:+#:=#:#:
Testing structurally possible combinat

In [8]:
filtered_data.token_len.describe()

count     927.000000
mean     1314.468177
std       621.975214
min       174.000000
25%       816.500000
50%      1204.000000
75%      1876.500000
max      2592.000000
Name: token_len, dtype: float64

In [9]:
len(filtered_data[filtered_data.token_len <= 7500])#.loc[ 751].generated_cot

927

In [10]:
print(filtered_data.loc[9416].generated_cot)

First, we examine the operators in the equations.
Let's look at the target expression and its operator, which is usually located in the middle.
There are no digits in the examples. This implies it is a cryptarithm; we need to decode the encrypted operations and values to calculate the result.
--- Решение ---
We model this as a cryptarithm: every visible non-operator symbol is one unique decimal digit, and multi-digit values cannot start with zero.
Parsed examples: '>:^@%=$]%@', '@@}]#=:/', '&]}]%=]#', '@$[$#=[%<'
Target expression: '$>}@/'
Symbols to decode: '#', '$', '%', '&', '/', ':', '<', '>', '@', ']'

Rule search
Evaluating operator '['
Examples: @$[$#=[%<
Testing structurally possible combinations:
 Config: standard
  - sub_signed -> format (sign_pref_symbol_raw) [MATCH]
 Config: little_endian
  - sub_signed -> format (sign_pref_symbol_rev)
Rule identified for '[': standard -> sub_signed -> sign_pref_symbol_raw
Verifying examples for this operator:
  @$ [ $# -> inputs A=26, B=67

In [11]:
print(filtered_data.loc[8488].prompt)

In Alice's Wonderland, a secret set of transformation rules is applied to equations. Below are a few examples:
:(*\| = \^}/
\/+\" = &}
&}-:| = (:
""+&^ = |"
!\*&| = &"&"
Now, determine the result for: (!*:"


In [12]:
text = """In Alice's Wonderland, a secret set of transformation rules is applied to equations. Below are a few examples:
[&+/` = '!
<\-<\ = \
`|-'' = -<|
/&*?\ = &/|
//-?? = -``
Now, determine the result for: &&+&`..."""

<>:3: SyntaxWarning: invalid escape sequence '\-'
<>:3: SyntaxWarning: invalid escape sequence '\-'
/tmp/ipykernel_330105/2259103573.py:3: SyntaxWarning: invalid escape sequence '\-'
  <\-<\ = \
